# 03 — Shot Prompting: Zero-Shot, One-Shot, and Few-Shot

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Zero-Shot Prompting

In [3]:
zero_shot = "Classify the sentiment of this review: 'The food was cold and the service was rude.'"
result = chat([{"role": "user", "content": zero_shot}], temperature=0, max_tokens=20)
print("Zero-shot sentiment classification:")
print(f"  Input : The food was cold and the service was rude.")
print(f"  Output: {result.strip()}")


Zero-shot sentiment classification:
  Input : The food was cold and the service was rude.
  Output: The sentiment of this review is negative. The reviewer expresses dissatisfaction with two aspects of their experience: the


## One-Shot Prompting

In [4]:
one_shot = (
    "Classify the sentiment of a review as POSITIVE, NEGATIVE, or NEUTRAL.\n\n"
    "Example:\n"
    "Review: \"Great product, fast delivery!\"\n"
    "Sentiment: POSITIVE\n\n"
    "Now classify:\n"
    "Review: \"The food was cold and the service was rude.\"\n"
    "Sentiment:"
)
result = chat([{"role": "user", "content": one_shot}], temperature=0, max_tokens=10)
print("One-shot result:", result.strip())


One-shot result: Sentiment: NEGATIVE

The review mentions two


## Few-Shot Prompting

In [5]:
few_shot = (
    "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL.\n\n"
    "Review: \"Absolutely love this! Best purchase ever.\"  -> POSITIVE\n"
    "Review: \"It broke after one day. Terrible quality.\"  -> NEGATIVE\n"
    "Review: \"It\'s okay, nothing special.\"               -> NEUTRAL\n"
    "Review: \"Fast shipping, product works as described.\" -> POSITIVE\n"
    "Review: \"Not what I expected, but not bad either.\"  -> NEUTRAL\n\n"
    "Review: \"The food was cold and the service was rude.\" ->"
)
result = chat([{"role": "user", "content": few_shot}], temperature=0, max_tokens=10)
print("Few-shot result:", result.strip())


Few-shot result: The sentiment of the review "The food was cold


## Few-Shot for Format Control

In [6]:
format_shots = (
    "Convert dates to ISO format.\n\n"
    "Input:  January 5th, 2023\n"
    "Output: 2023-01-05\n\n"
    "Input:  March 22, 2024\n"
    "Output: 2024-03-22\n\n"
    "Input:  December 1st, 2025\n"
    "Output:"
)
result = chat([{"role": "user", "content": format_shots}], temperature=0, max_tokens=15)
print("Format-controlled output:", result.strip())


Format-controlled output: Here's a Python function that converts dates to ISO format:

```python


## Few-Shot for Custom Classification

In [7]:
custom_clf = (
    "Classify customer support tickets into: BILLING, TECHNICAL, ACCOUNT, GENERAL.\n\n"
    "Ticket: \"I was charged twice this month.\"          -> BILLING\n"
    "Ticket: \"App crashes when I open settings.\"        -> TECHNICAL\n"
    "Ticket: \"I forgot my password and can\'t log in.\" -> ACCOUNT\n"
    "Ticket: \"What are your business hours?\"            -> GENERAL\n"
    "Ticket: \"My subscription renewed but features are not working.\" ->"
)
result = chat([{"role": "user", "content": custom_clf}], temperature=0, max_tokens=15)
print("Custom classification:", result.strip())


Custom classification: Based on the given information, I would classify the last ticket as follows:


## Comparison: 0-shot vs 3-shot

In [8]:
task_text = "Alice will send the report by Friday. Bob needs to review the budget. Team should schedule a follow-up for next week."

zero = chat([{"role": "user", "content": f"Extract all action items as a bullet list:\n{task_text}"}],
            temperature=0, max_tokens=100)

few = (
    "Extract all action items as a bullet list.\n\n"
    "Meeting: John will call the client Monday. Sarah updates the slides.\n"
    "Action items:\n"
    "- John: call client (Monday)\n"
    "- Sarah: update slides\n\n"
    f"Meeting: {task_text}\n"
    "Action items:"
)
few_result = chat([{"role": "user", "content": few}], temperature=0, max_tokens=100)

print("Zero-shot:")
print(zero.strip())
print()
print("Few-shot:")
print(few_result.strip())


Zero-shot:
Here are the action items as a bullet list:

* Alice will send the report by Friday.
* Bob needs to review the budget.
* Team should schedule a follow-up for next week.

Few-shot:
Here are the action items extracted as a bullet list:

- John: call client (Monday)
- Sarah: update slides
- Alice: send report by Friday
- Bob: review budget
- Team: schedule follow-up for next week
